In [1]:
from transformers import AudioFlamingo3ForConditionalGeneration, AutoProcessor
import pandas as pd

In [2]:
model_id = "nvidia/music-flamingo-hf"
processor = AutoProcessor.from_pretrained(model_id)
model = AudioFlamingo3ForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    offload_buffers=True)


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/830 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.


In [3]:
conversation = [
    {
        "role": "user",
        "content": [
            {"type": "text",
             "text": """For each of the following emotions, rate the intensity (ranging from 0 to 100) you percieve in this music excerpt.
- Wonder (Filled with wonder, Dazzled, Allured, Moved)
- Transcendence (Fascinated, Overwhelmed, Feelings of transcendence and spirituality)
- Nostalgia (Nostalgic, Dreamy, Sentimental, Melancholic)
- Tenderness (Tender, Affectionate, In love, Mellowed)
- Peacefulness (Serene, Calm, Soothed, Relaxed)
- Joy (Joyful, Amused, Animated, Bouncy)
- Power (Strong, Triumphant, Energetic, Fiery)
- Tension (Tense, Agitated, Nervous, Irritated)
- Sadness (Sad, Sorrowful)
            """},
            {"type": "audio", "path": "tracks/0il663f3f63cvsRJtGmdO2.mp3"},
        ],
    }
]

inputs = processor.apply_chat_template(
    conversation,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=256)

decoded_outputs = processor.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)
print(decoded_outputs)

['- Wonder: 70\n- Transcendence: 50\n- Nostalgia: 20\n- Tenderness: 10\n- Peacefulness: 20\n- Joy: 80\n- Power: 90\n- Tension: 10\n- Sadness: 0']


In [3]:
df = pd.read_csv('train.csv')

conversation = []

for i, row in df.iterrows():

    track_info = [
            {
                "role": "user",
                "content": [
                    {"type": "text",
                     "text": """For each of the following emotions, rate the intensity (ranging from 0 to 100) you percieve in this music excerpt.
- Wonder (Filled with wonder, Dazzled, Allured, Moved)
- Transcendence (Fascinated, Overwhelmed, Feelings of transcendence and spirituality)
- Nostalgia (Nostalgic, Dreamy, Sentimental, Melancholic)
- Tenderness (Tender, Affectionate, In love, Mellowed)
- Peacefulness (Serene, Calm, Soothed, Relaxed)
- Joy (Joyful, Amused, Animated, Bouncy)
- Sadness (Sad, Sorrowful)
- Power (Strong, Triumphant, Energetic, Fiery)
- Tension (Tense, Agitated, Nervous, Irritated)
                    """},
                    {"type": "audio",
                     "path": row['path']},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text",
                             "text": f"- Wonder: {row['Wonder']}\n- Transcendence: {row['Transcendence']}\n- Nostalgia: {row['Nostalgia']}\n- Tenderness: {row['Tenderness']}\n- Peacefulness: {row['Peacefulness']}\n- Joy: {row['Joy']}\n- Sadness: {row['Sadness']}\n- Power: {row['Power']}\n- Tension: {row['Tension']}"}],
            }
        ]

    conversation.append(track_info)

conversation

[[{'role': 'user',
   'content': [{'type': 'text',
     'text': 'For each of the following emotions, rate the intensity (ranging from 0 to 100) you percieve in this music excerpt.\n- Wonder (Filled with wonder, Dazzled, Allured, Moved)\n- Transcendence (Fascinated, Overwhelmed, Feelings of transcendence and spirituality)\n- Nostalgia (Nostalgic, Dreamy, Sentimental, Melancholic)\n- Tenderness (Tender, Affectionate, In love, Mellowed)\n- Peacefulness (Serene, Calm, Soothed, Relaxed)\n- Joy (Joyful, Amused, Animated, Bouncy)\n- Sadness (Sad, Sorrowful)\n- Power (Strong, Triumphant, Energetic, Fiery)\n- Tension (Tense, Agitated, Nervous, Irritated)\n                    '},
    {'type': 'audio', 'path': 'tracks/7i2DJ88J7jQ8K7zqFX2fW8.mp3'}]},
  {'role': 'assistant',
   'content': [{'type': 'text',
     'text': '- Wonder: 10.69\n- Transcendence: 14.77\n- Nostalgia: 7.92\n- Tenderness: 20.69\n- Peacefulness: 18.54\n- Joy: 17.0\n- Sadness: 9.92\n- Power: 16.15\n- Tension: 14.23'}]}],
 [{'role

In [4]:
model.train()

inputs = processor.apply_chat_template(
    conversation,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    output_labels=True,
).to(model.device)

loss = model(**inputs).loss
loss.backward()

OutOfMemoryError: CUDA out of memory. Tried to allocate 7.98 GiB. GPU 0 has a total capacity of 15.93 GiB of which 0 bytes is free. Of the allocated memory 27.11 GiB is allocated by PyTorch, and 2.40 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)